# Shor to Regev: task-aware QFT precision

This standalone notebook demonstrates the new additive study without modifying or rerunning the repository's frozen Regev holdout. It starts with a complete small-scale Shor pipeline, then loads the frozen Shor and Regev result records to compare QFT approximation at circuit, state, measurement, and decoder levels.

## The question

A worst-case matrix bound treats every possible QFT input and every downstream event adversarially. Shor and Regev prepare structured arithmetic states and use very different decoders: continued fractions versus augmented-lattice/LLL recovery. The study asks whether the same style of QFT truncation can therefore have different task-level consequences.

In [1]:
from pathlib import Path
import json
import sys
import pandas as pd
import matplotlib.pyplot as plt

cwd = Path.cwd().resolve()
if cwd.name == 'shor_to_regev_study':
    STUDY, REPOSITORY = cwd, cwd.parent
else:
    REPOSITORY, STUDY = cwd, cwd / 'shor_to_regev_study'
sys.path.insert(0, str(REPOSITORY))
from shor_to_regev_study.shor import shor_factor
print('Repository:', REPOSITORY)
print('Study folder:', STUDY)

Repository: /Users/summermalik/Desktop/regevs
Study folder: /Users/summermalik/Desktop/regevs/shor_to_regev_study


## 1. Complete standard Shor pipeline

The implementation follows `N, base -> gcd shortcut -> modular exponentiation -> inverse QFT -> measured phase -> continued fractions -> modular order verification -> gcd factor extraction`. The decoder never receives a true order or factors.

In [2]:
examples = pd.read_csv(STUDY / 'results' / 'shor_experiments' / 'examples.csv')
examples[['N', 'base', 'recovered_order', 'factor_success', 'factor_pair', 'failure_reason']]

,N,base,recovered_order,factor_success,factor_pair,failure_reason
0,15,2,4,True,"[3, 5]",NaN
1,15,14,2,False,NaN,trivial_square_root
2,21,2,6,True,"[3, 7]",NaN
3,21,4,3,False,NaN,odd_order
4,21,5,6,False,NaN,trivial_square_root
5,33,10,2,True,"[3, 11]",NaN


In [3]:
live = shor_factor(21, base=2, shots=16, seed=2026)
{
    'N': live.N,
    'base': live.base,
    'verified_order': live.recovered_order,
    'verified_factor_pair': live.factor_pair,
    'gcd_shortcut': live.gcd_shortcut,
    'decoder_received_true_order_or_factors': False,
}

{'N': 21,
 'base': 2,
 'verified_order': 6,
 'verified_factor_pair': (3, 7),
 'gcd_shortcut': False,
 'decoder_received_true_order_or_factors': False}

Successful examples include `(15,2)->(3,5)`, `(21,2)->(3,7)`, and `(33,10)->(3,11)`. Bases `(21,4)` and `(21,5)` deliberately demonstrate odd-order and trivial-square-root failure.

## 2. Frozen Shor QFT robustness holdout

The primary unit is one of eight held-out `(N,base)` instances. The grid uses 4, 8, or 16 shots; zero or 1% independent per-qubit readout flips; exact QFT through three omitted phase layers; 64 paired batches; and 5,000 whole-instance bootstrap draws.

In [4]:
shor_root = STUDY / 'results' / 'shor_qft_robustness'
shor_configuration = json.loads((shor_root / 'configuration.json').read_text())
shor_configuration['heldout_instances'], shor_configuration['row_counts']

([[35, 2], [35, 11], [39, 2], [51, 2], [55, 2], [65, 3], [77, 8], [91, 9]],
 {'bootstrap_draw_rows': 120000,
  'configuration_rows': 192,
  'leave_one_instance_out_rows': 144,
  'margin_sensitivity_rows': 96,
  'margin_summary_rows': 24,
  'paired_exact_test_rows': 36,
  'paired_rows': 24,
  'per_instance_rows': 192,
  'trial_rows': 12288})

In [5]:
paired = pd.read_csv(shor_root / 'paired_rows.csv')
primary = paired[(paired.shots == 8) & (paired.readout_bitflip_probability == 0)]
primary[['omitted_layers', 'order_mean_difference', 'factor_mean_difference', 'order_ci95_low', 'factor_ci95_low', 'noninferior_at_0_10']]

,omitted_layers,order_mean_difference,factor_mean_difference,order_ci95_low,factor_ci95_low,noninferior_at_0_10
8,0,0.0,0.0,0.0,0.0,True
9,1,0.0,0.0,0.0,0.0,True
10,2,0.0,0.0,0.0,0.0,True
11,3,0.0,0.0,0.0,0.0,True


No paired order or factor batch decision changed anywhere in the frozen grid. At three omitted layers the worst-case certificate rejects all eight primary instances, while exact distribution TV is at most `2.67e-5` and the direct finite distribution certificate approves all eight. The truncation removes six controlled phases and 12 QFT-only CX gates, but no isolated-QFT depth saving was observed.

In [6]:
configuration_rows = pd.read_csv(shor_root / 'configuration_rows.csv')
configuration_rows.groupby('omitted_layers').agg(
    worst_case_approvals=('worst_case_certified', 'sum'),
    distribution_approvals=('distribution_certified', 'sum'),
    maximum_tv=('distribution_tv', 'max'),
    maximum_cp_saving=('controlled_phase_saving', 'max'),
    maximum_cx_saving=('compiled_cx_saving', 'max'),
)

,worst_case_approvals,distribution_approvals,maximum_tv,maximum_cp_saving,maximum_cx_saving
omitted_layers,,,,,
0,48,48,0.000000e+00,0,0
1,48,48,6.253785e-07,1,2
2,12,48,4.791266e-06,3,6
3,0,48,2.662201e-05,6,12


## 3. Cross-decoder result

The existing Regev result files are loaded read-only. A common schema records worst-case and state-specific certificate decisions, measured TV, verified task probability, exact-QFT task probability, task difference, and QFT-only resources for Shor order/factor and Regev `L\L0`/factor endpoints.

In [7]:
cross_root = STUDY / 'results' / 'cross_algorithm_precision'
outcome = json.loads((cross_root / 'outcome.json').read_text())
outcome

{'meaning': 'Both Shor and Regev show held-out task robustness beyond the worst-case certificate.',
 'outcome': 'A',
 'regev_uncertified_noninferior_truncation': True,
 'shor_uncertified_noninferior_truncation': True}

**Outcome A:** both continued-fraction and augmented-lattice recovery have held-out truncations that remain within their declared task-loss rule after the worst-case certificate rejects. Shor's tested gap is fully explained by a direct distribution certificate; not every empirically non-inferior Regev cell is uniformly closed by that certificate.

In [8]:
summary = pd.read_csv(cross_root / 'algorithm_summary_rows.csv')
summary[['algorithm', 'endpoint', 'approximation_level', 'instances', 'mean_task_difference', 'mean_measured_tv', 'worst_case_certified_fraction', 'state_specific_certified_fraction']]

,algorithm,endpoint,approximation_level,instances,mean_task_difference,mean_measured_tv,worst_case_certified_fraction,state_specific_certified_fraction
0,Regev,L_minus_L0,0,48,0.000000,0.000000e+00,1.00,1.000000
1,Regev,L_minus_L0,1,48,-0.004232,2.058792e-02,0.00,0.458333
2,Regev,L_minus_L0,2,48,-0.053060,1.115242e-01,0.00,0.104167
3,Regev,L_minus_L0,3,32,-0.094238,2.765458e-01,0.00,0.062500
4,Regev,L_minus_L0,4,16,-0.111328,4.059647e-01,0.00,0.062500
5,Regev,factor,0,48,0.000000,0.000000e+00,1.00,1.000000
6,Regev,factor,1,48,-0.004232,2.058792e-02,0.00,0.458333
7,Regev,factor,2,48,-0.053060,1.115242e-01,0.00,0.104167
8,Regev,factor,3,32,-0.094238,2.765458e-01,0.00,0.062500
9,Regev,factor,4,16,-0.111328,4.059647e-01,0.00,0.062500


## 4. Decoder-boundary predictor: negative mechanism result

The intended strong mechanism did not survive: the heterogeneous decoder-boundary proxy did not outperform measured TV or omitted-layer accuracy. The factor-blind logistic surrogate improves dramatically over the worst-case certificate but still makes false approvals and therefore is not a certificate.

In [9]:
evaluation = pd.read_csv(cross_root / 'predictor_evaluation_rows.csv')
evaluation[['method', 'accuracy', 'brier', 'false_approvals', 'false_rejections', 'precision', 'recall']].sort_values('accuracy', ascending=False)

,method,accuracy,brier,false_approvals,false_rejections,precision,recall
3,omitted_layers_alone,0.906250,0.093750,27,0,0.906250,1.000000
0,factor_blind_logistic,0.902778,0.082341,10,18,0.960474,0.931034
2,measured_tv_alone,0.895833,0.104167,6,24,0.975309,0.908046
4,decoder_boundary_proxy_alone,0.885417,0.114583,27,6,0.904255,0.977011
1,worst_case_certificate,0.302083,0.697917,0,201,1.000000,0.229885


In [10]:
figure, axis = plt.subplots(figsize=(8, 4))
plot_data = evaluation.sort_values('accuracy')
axis.barh(plot_data.method, plot_data.accuracy)
axis.set_xlim(0, 1)
axis.set_xlabel('Held-out accuracy')
axis.set_title('Factor-blind precision decisions are useful but imperfect')
plt.tight_layout()
plt.show()

/var/folders/tv/_11343vn6153sfykk5r11dq40000gn/T/ipykernel_37244/2652902543.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Publication-safe conclusion

This is a reproducible finite Shor-to-Regev testbed, not a new factoring algorithm or hardware speedup. Both decoders tolerate limited truncations rejected by a worst-case QFT certificate, but their finite state/distribution mechanisms differ. Direct distribution analysis completely explains the tested Shor gap and only part of the Regev gap; a universal decoder-boundary predictor remains open.

## Reproduce

From the repository root:

```bash
MPLCONFIGDIR=/tmp/mpl PYTHONPATH=. .venv/bin/python shor_to_regev_study/run_shor_experiments.py
MPLCONFIGDIR=/tmp/mpl PYTHONPATH=. .venv/bin/python shor_to_regev_study/run_shor_qft_robustness.py
MPLCONFIGDIR=/tmp/mpl PYTHONPATH=. .venv/bin/python shor_to_regev_study/run_cross_algorithm_precision_study.py
MPLCONFIGDIR=/tmp/mpl PYTHONPATH=. .venv/bin/python -m pytest -q
```